# GIAI ĐOẠN 1: THU THẬP DỮ LIỆU ĐÁNH GIÁ (CRAWL DMX REVIEWS)

## 1. Bản chất bài toán
- **Bài toán**: Phân loại cảm xúc khách hàng (Sentiment Analysis) thành 3 nhóm: `Tiêu cực (1-2 sao)`, `Trung tính (3 sao)`, `Tích cực (4-5 sao)`.
- **Mục tiêu**: Thu thập đúng **$N$ review** bất kỳ phục vụ demo hoặc báo cáo mô hình.
- **Chiến lược cào nhanh & Tự nhiên (Phương án 3 kết hợp)**:
  1. **Tỉ lệ phân bố thực tế (60% Tích cực - 20% Trung tính - 20% Tiêu cực)**: Phản ánh đúng hành vi khách hàng trên sàn TMĐT thực tế, giúp cào cực nhanh mà không bị nghẽn ở các lớp hiếm.
  2. **Ưu tiên sản phẩm nhiều review đa chiều**: File `product_ids.csv` đã được sắp xếp ưu tiên các sản phẩm có nhiều đánh giá 1-3 sao lên đầu.
  3. **An toàn khi dừng (Ctrl + C / Interrupt Kernel)**: Dù chạy xong hay bấm dừng giữa chừng, toàn bộ review đã thu thập vẫn được tự động lưu ra file CSV và in bảng báo cáo tổng hợp.

### Bước 1: Thiết lập tham số và nạp danh sách sản phẩm

In [ ]:
import asyncio
import math
from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright

# =============================================================================
# CẤU HÌNH THAM SỐ CRAWL
# =============================================================================
N_DESIRED = 1000  # Đặt số lượng review mong muốn (50, 500, 1000, 2000...)
OUTPUT_CSV = Path(f"../data/raw/reviews_{N_DESIRED}.csv")
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Nạp kho sản phẩm đã xếp thứ tự ưu tiên
df_products = pd.read_csv("../data/product_ids.csv")
product_pool = df_products["product_id"].tolist()

# Phân chia hạn ngạch theo tỉ lệ thực tế: 20% Tiêu cực - 20% Trung tính - 60% Tích cực
target_tieu_cuc = int(N_DESIRED * 0.20)
target_trung_tinh = int(N_DESIRED * 0.20)
target_tich_cuc = N_DESIRED - target_tieu_cuc - target_trung_tinh

TARGET_QUOTAS = {
    "Tiêu cực":   {"scores": ["1", "2"], "target": target_tieu_cuc},
    "Trung tính": {"scores": ["3"],       "target": target_trung_tinh},
    "Tích cực":   {"scores": ["4", "5"], "target": target_tich_cuc}
}

print(f"-> Mục tiêu: {N_DESIRED} reviews (Theo tỉ lệ thực tế: 20% Tiêu cực - 20% Trung tính - 60% Tích cực)")
for k, v in TARGET_QUOTAS.items():
    percent = (v['target'] / N_DESIRED) * 100
    print(f"   * {k:<10}: {v['target']:<4} reviews ({percent:.0f}%)")

### Bước 2: Hàm bóc tách dữ liệu từ HTML

In [ ]:
def parse_reviews_from_html(html_text: str) -> list[dict]:
    """
    Bóc tách các thẻ đánh giá từ HTML trả về của API PagingAllRating.
    """
    soup = BeautifulSoup(html_text or "", "html.parser")
    results = []
    
    for li in soup.select("li.par"):
        r_id = (li.get("id") or "").replace("r-", "").strip()
        if not r_id.isdigit():
            continue
            
        stars = len(li.select(".cmt-top-star i.iconcmt-starbuy"))
        content_tag = li.select_one(".cmt-txt")
        content = content_tag.get_text(strip=True) if content_tag else ""
        
        if content:
            results.append({
                "review_id": r_id,
                "so_sao": stars,
                "noi_dung": content
            })
            
    return results

### Bước 3: Tiến hành Crawl (Hỗ trợ dừng bất kỳ lúc nào mà không mất dữ liệu)

In [ ]:
collected_reviews = []
seen_review_ids = set()
group_counts = {group: 0 for group in TARGET_QUOTAS}

print("Đang thu thập dữ liệu (Bấm nút Stop/Interrupt bất kỳ lúc nào nếu muốn dừng sớm):\n")

try:
    async with async_playwright() as p:
        browser = await p.chromium.launch(channel="chrome", headless=True)
        tab = await browser.new_page()
        await tab.goto("https://www.dienmayxanh.com", wait_until="domcontentloaded")
        
        for pid in product_pool:
            if all(group_counts[g] >= TARGET_QUOTAS[g]["target"] for g in TARGET_QUOTAS):
                break
                
            for group_name, cfg in TARGET_QUOTAS.items():
                target = cfg["target"]
                if group_counts[group_name] >= target:
                    continue
                    
                for score in cfg["scores"]:
                    if group_counts[group_name] >= target:
                        break
                        
                    page = 1
                    while group_counts[group_name] < target:
                        response_html = await tab.evaluate("""
                            async ([productId, pageNum, scoreFilter]) => {
                                const res = await fetch('https://www.dienmayxanh.com/comment/PagingAllRating', {
                                    method: 'POST',
                                    headers: {'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8'},
                                    body: `productid=${productId}&page=${pageNum}&score=${scoreFilter}&pagesize=20`
                                });
                                return await res.text();
                            }
                        """, [pid, page, score])
                        
                        batch = parse_reviews_from_html(response_html)
                        if not batch:
                            break
                            
                        added = 0
                        for item in batch:
                            if item["review_id"] not in seen_review_ids and group_counts[group_name] < target:
                                seen_review_ids.add(item["review_id"])
                                item["product_id"] = pid
                                item["nhom_cam_xuc"] = group_name
                                collected_reviews.append(item)
                                group_counts[group_name] += 1
                                added += 1
                                
                        if added == 0:
                            break
                        page += 1
                        await asyncio.sleep(0.3)
            
            print(f"Product_id: {pid:<8} | Tiêu cực: {group_counts['Tiêu cực']}/{TARGET_QUOTAS['Tiêu cực']['target']:<3} | Trung tính: {group_counts['Trung tính']}/{TARGET_QUOTAS['Trung tính']['target']:<3} | Tích cực: {group_counts['Tích cực']}/{TARGET_QUOTAS['Tích cực']['target']:<3}")

        await browser.close()

except (KeyboardInterrupt, asyncio.CancelledError):
    print("\n[!] Đã nhận lệnh dừng từ người dùng (KeyboardInterrupt / Stop Kernel).")

finally:
    if collected_reviews:
        df_result = pd.DataFrame(collected_reviews)
        df_result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
        
        print("\n" + "="*50)
        print("BÁO CÁO TỔNG HỢP KẾT QUẢ CRAWL:")
        print(f"- Tổng số review đã thu thập được: {len(df_result)}/{N_DESIRED}")
        print(f"- File lưu trữ: {OUTPUT_CSV.resolve()}")
        print("- Phân bố thực tế các lớp cảm xúc:")
        for group, count in df_result["nhom_cam_xuc"].value_counts().items():
            print(f"  + {group:<10}: {count} reviews ({count/len(df_result)*100:.1f}%)")
        print("="*50)
    else:
        print("\n[!] Chưa thu thập được review nào trước khi dừng.")